In [2]:
import importlib.util
import subprocess
import sys

required = ["pandas", "pygwalker", "pyodbc", "dotenv", "sqlalchemy"]
missing = [package for package in required if importlib.util.find_spec(package) is None]

if missing:
    print(f"Paquetes faltantes en el kernel actual: {missing}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet",
         "pandas", "pygwalker", "sqlalchemy", "pyodbc", "python-dotenv"]
    )
    print("Instalación completada.")
else:
    print("Todos los paquetes requeridos ya están disponibles en este kernel.")


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import importlib
import os
from pathlib import Path

# Dependencias requeridas para la notebook
REQUIRED_PACKAGES = ["pandas", "pygwalker", "pyodbc", "dotenv", "sqlalchemy"]


def ensure_packages():
    missing = []
    for module_name in REQUIRED_PACKAGES:
        try:
            importlib.import_module(module_name)
        except ModuleNotFoundError:
            missing.append(module_name)

    if missing:
        print(f"Instalando dependencias faltantes: {missing}")
        %pip install --quiet pandas pygwalker sqlalchemy pyodbc python-dotenv
        for module_name in missing:
            importlib.import_module(module_name)


ensure_packages()

# VS Code/pyright puede marcar estas importaciones como no resueltas
# si el kernel activo no coincide con el entorno donde se instalaron.
# Se ignora el warning específico de resolución del módulo para no bloquear la ejecución.
import pandas as pd  # pyright: ignore[reportMissingImports, reportMissingModuleSource]
import pygwalker as pyg  # pyright: ignore[reportMissingImports, reportMissingModuleSource]
import pyodbc  # pyright: ignore[reportMissingImports, reportMissingModuleSource]
from dotenv import load_dotenv  # pyright: ignore[reportMissingImports, reportMissingModuleSource]
from sqlalchemy import create_engine, text  # pyright: ignore[reportMissingImports, reportMissingModuleSource]


def find_env_file():
    base_dir = Path.cwd().resolve()
    candidates = [
        base_dir / ".env",
        base_dir.parent / ".env",
        base_dir / ".." / ".env",
        Path("../.env"),
    ]
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved.is_file():
            return resolved
    return None


def build_sample_logistics():
    return pd.DataFrame(
        {
            "fecha": pd.date_range("2024-01-01", periods=10, freq="D"),
            "region": ["Norte", "Sur", "Centro", "Norte", "Sur", "Centro", "Norte", "Sur", "Centro", "Norte"],
            "ruta": ["R-101", "R-202", "R-303", "R-101", "R-202", "R-303", "R-101", "R-202", "R-303", "R-101"],
            "envios": [120, 95, 140, 132, 110, 150, 125, 105, 145, 136],
            "tiempo_promedio_horas": [18.5, 23.4, 20.1, 19.2, 22.1, 21.5, 18.8, 23.1, 20.7, 19.4],
            "coste_total": [3500, 3100, 4200, 3650, 3180, 4400, 3600, 3220, 4300, 3750],
        }
    )


def build_sample_performance():
    return pd.DataFrame(
        {
            "categoria": ["Electrónicos", "Hogar", "Ropa", "Deportes", "Jardín", "Belleza", "Muebles", "Tecnología"],
            "ventas": [120000, 98000, 76000, 87000, 54000, 47000, 66000, 93000],
            "margen": [0.24, 0.19, 0.22, 0.17, 0.15, 0.18, 0.20, 0.23],
            "clientes": [420, 380, 290, 310, 225, 210, 260, 340],
            "satisfaccion": [4.5, 4.2, 4.1, 4.3, 3.9, 4.0, 4.2, 4.4],
        }
    )


# 1. Carga explícita y robusta del entorno
env_path = find_env_file()
if env_path is None:
    raise RuntimeError(
        "❌ No se encontró el archivo .env en la raíz esperada. "
        "Revisa que exista en P4_Real_World_Ingestion/.env"
    )

# Se limpia para evitar que variables previas sobrescriban el entorno actual.
for key in ["DB_USER", "DB_PASSWORD"]:
    os.environ.pop(key, None)

load_dotenv(dotenv_path=env_path, override=True)
print(f"✅ Variables de entorno cargadas desde: {env_path}")

server = (os.getenv("DB_SERVER") or "").strip()
database = (os.getenv("DB_NAME") or "").strip()
driver = (os.getenv("DB_DRIVER") or "ODBC Driver 17 for SQL Server").strip()
db_user = (os.getenv("DB_USER") or "").strip()
db_password = (os.getenv("DB_PASSWORD") or "").strip()
use_trusted = os.getenv("DB_USE_TRUSTED_CONNECTION", "false").strip().lower() in {"1", "true", "yes", "y"}

if (not db_user or not db_password) and not use_trusted:
    use_trusted = True
    print("ℹ️ No se detectaron DB_USER/DB_PASSWORD; se usará autenticación de Windows por defecto.")

if not server or not database:
    raise RuntimeError(
        "❌ Faltan credenciales críticas (DB_SERVER o DB_NAME). Revisa tu archivo .env."
    )

# Valida que el driver exista y usa un fallback si el 17 no está instalado.
available_drivers = [d for d in pyodbc.drivers()]
if driver not in available_drivers:
    fallback_drivers = ["ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server", "SQL Server"]
    for candidate_driver in fallback_drivers:
        if candidate_driver in available_drivers:
            driver = candidate_driver
            break

if driver not in available_drivers:
    raise RuntimeError(
        "❌ No se encontró un driver ODBC de SQL Server instalado. "
        "Instala 'ODBC Driver 17 for SQL Server' o 'ODBC Driver 18 for SQL Server'.\n"
        f"Drivers disponibles: {available_drivers}"
    )

# Construcción segura de la cadena de conexión.
conn_parts = [
    f"DRIVER={{{driver}}}",
    f"SERVER={server}",
    f"DATABASE={database}",
    "Encrypt=yes",
    "TrustServerCertificate=yes",
    "Connection Timeout=30",
]

if db_user and db_password:
    conn_parts.append(f"UID={db_user}")
    conn_parts.append(f"PWD={db_password}")
    auth_mode = "SQL Server Authentication"
elif use_trusted:
    conn_parts.append("Trusted_Connection=yes")
    auth_mode = "Windows Authentication"
else:
    raise RuntimeError(
        "❌ Faltan credenciales. Define DB_USER y DB_PASSWORD para autenticación SQL, "
        "o habilita DB_USE_TRUSTED_CONNECTION=true para autenticación de Windows en el .env."
    )

conn_string = ";".join(conn_parts) + ";"
print(f"🔐 Intentando conectar con {auth_mode} a {server}/{database} usando driver {driver}...")

engine = create_engine(
    "mssql+pyodbc://",
    creator=lambda: pyodbc.connect(conn_string, timeout=30, autocommit=True),
)

# 2. Validación de conexión y carga de datasets
sample_logistics = build_sample_logistics()
sample_performance = build_sample_performance()
df_logistics = None
df_performance = None

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1")).fetchone()
    print("✅ Conexión a SQL Server correcta.")
    try:
        print("📊 Extrayendo datos desde SQL Server...")
        df_logistics = pd.read_sql(text("SELECT * FROM Analytics.vw_Shipping_Efficiency"), engine)
        df_performance = pd.read_sql(text("SELECT * FROM Analytics.vw_Category_Performance"), engine)
    except Exception as exc:
        raise RuntimeError(
            f"""
            ❌ ERROR CONSULTANDO VISTAS ANALÍTICAS

            Vista:
            Analytics.vw_Shipping_Efficiency

            Vista:
            Analytics.vw_Category_Performance

            Detalle:
            {exc}
            """
        ) from exc

except Exception as exc:
    if use_trusted:
        current_user = os.getlogin() if os.name == "nt" else "el usuario actual del sistema"
        user_hint = (
            f"Si usas autenticación de Windows, revisa que '{current_user}' exista como Login "
            "en SQL Server y tenga permisos sobre la base de datos. "
            f"Además confirma que la base de datos '{database}' exista en el servidor '{server}'."
        )
    else:
        user_hint = (
            f"Revisa que el usuario '{db_user}' exista como Login en SQL Server, que la contraseña sea correcta "
            f"y que el usuario tenga permisos sobre la base de datos '{database}'."
        )

    raise RuntimeError(
        f"""
❌ ERROR DE CONEXIÓN A SQL SERVER

Servidor : {server}
Base     : {database}

Detalle:
{exc}

Sugerencia:
{user_hint}
"""
    ) from exc

# 3. Validación final de datasets
if df_logistics is None or df_logistics.empty:
    raise ValueError("❌ ERROR: El DataFrame de logística está vacío. Revisa la vista en SQL.")
if df_performance is None or df_performance.empty:
    raise ValueError("❌ ERROR: El DataFrame de rendimiento está vacío. Revisa la vista en SQL.")

print("✅ DataFrames listos para la dashboard.")
print(f"Logística: {df_logistics.shape[0]} filas, {df_logistics.shape[1]} columnas")
print(f"Rendimiento: {df_performance.shape[0]} filas, {df_performance.shape[1]} columnas")
print(df_logistics.head())
print(df_performance.head())


✅ Variables de entorno cargadas desde: D:\01.Datos_AlbertoDzib\Portafolio-Alberto\P4_Real_World_Ingestion\.env
🔐 Intentando conectar con Windows Authentication a DZIBPC/P4_Global_SupplyChain usando driver ODBC Driver 17 for SQL Server...
✅ Conexión a SQL Server correcta.
📊 Extrayendo datos desde SQL Server...
⚠️ La conexión a SQL Server sí funciona, pero las vistas analíticas no están disponibles. Se usará un conjunto de datos de ejemplo para continuar.
Detalle técnico: Execution failed on sql 'SELECT * FROM Analytics.vw_Shipping_Efficiency': (pyodbc.ProgrammingError) ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'Analytics.vw_Shipping_Efficiency'. (208) (SQLExecDirectW)")
[SQL: SELECT * FROM Analytics.vw_Shipping_Efficiency]
(Background on this error at: https://sqlalche.me/e/21/f405)
✅ DataFrames listos para la dashboard.
Logística: 10 filas, 6 columnas
Rendimiento: 8 filas, 5 columnas
       fecha  region   ruta  envios  tiempo_promedio

In [8]:
from pathlib import Path
p = Path(r"D:\01.Datos_AlbertoDzib\Portafolio-Alberto\P4_Real_World_Ingestion\.env")
print(p.read_text(encoding="utf-8"))


DB_SERVER=DZIBPC
DB_NAME=P4_Global_SupplyChain
DB_DRIVER=ODBC Driver 17 for SQL Server

